# 🎬 LTX-Video 8-Segundos Studio (Google Colab Pro)
Pipeline optimizado para generación nativa de videos de **8 segundos (~193 frames @ 24fps)** con **LTX-Video (Lightricks)**.

### ⚡ Especificaciones Técnicas:
- **Modelo:** `Lightricks/LTX-Video` (DiT ~2B parámetros)
- **Duración:** 8.04 segundos (193 cuadros a 24 fps)
- **Aceleración:** PyTorch SDPA / Flash-Attention / bfloat16 con offloading
- **Túnel seguro:** **Cloudflare Quick Tunnel** (sin cuentas, sin ngrok)
- **Persistencia:** Guardado automático y caché de pesos opcional en Google Drive

## 1. Verificación de Hardware (GPU)
Comprueba que tengas asignada una GPU **L4 (Recomendada)** o **A100** en Colab Pro.

In [ ]:
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise SystemError("⚠️ No se detectó GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno' y selecciona GPU (L4 o A100).")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU activa: {gpu_name} ({vram_gb:.1f} GB VRAM)")

## 2. Montar Google Drive (Persistencia de Videos y Modelos)
Permite que los videos generados se guarden directamente en tu Google Drive y que los pesos del modelo queden en caché para no volver a descargarlos en futuras sesiones.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Crear carpetas de trabajo en Drive
OUTPUT_DIR = "/content/drive/MyDrive/LTX_Video_Outputs"
CACHE_DIR = "/content/drive/MyDrive/LTX_Video_Cache"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Configurar Hugging Face cache en Drive si deseas persistencia de pesos
# os.environ["HF_HOME"] = CACHE_DIR

print(f"📂 Salidas guardadas en: {OUTPUT_DIR}")
print(f"📦 Caché de modelos: {CACHE_DIR}")

## 3. Instalación de Dependencias y Cloudflare Tunnel
Instalamos `diffusers` optimizado, dependencias de aceleración y descargamos el binario oficial de `cloudflared`.

In [ ]:
# 1. Instalar bibliotecas de IA
!pip install -q --upgrade pip
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate sentencepiece imageio-ffmpeg gradio einops torchvision

# 2. Descargar binario de Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb

!cloudflared --version
print("✅ Dependencias y Cloudflare Tunnel instalados exitosamente.")

## 4. Carga del Pipeline LTX-Video en VRAM
Cargamos `LTXPipeline` en `bfloat16` con `enable_model_cpu_offload()` para garantizar que la generación de 8 segundos nunca cause OOM.

In [ ]:
import torch
from diffusers import LTXPipeline

MODEL_ID = "Lightricks/LTX-Video"
print(f"⏳ Cargando modelo {MODEL_ID} en bfloat16...")

pipe = LTXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
)

# Optimizaciones críticas de memoria
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

print("🚀 Pipeline listo para inferencia ultra-rápida.")

## 5. Servidor Gradio + Túnel Cloudflare
Lanza la interfaz web y genera una URL pública `.trycloudflare.com` mediante túnel seguro sin cuentas de terceros.

In [ ]:
import os
import time
import re
import subprocess
import gradio as gr
import torch
from diffusers.utils import export_to_video
from datetime import datetime

LOCAL_PORT = 7860

def generate_video(
    prompt,
    negative_prompt,
    resolution,
    duration_preset,
    num_inference_steps,
    guidance_scale,
    seed
):
    if not prompt or prompt.strip() == "":
        return None, "❌ Por favor ingresa un prompt descriptivo."
    
    # Mapeo de resolución (debe ser múltiplo de 32)
    res_dict = {
        "768 x 512 (Horizontal 16:9 aprox)": (768, 512),
        "512 x 768 (Vertical 9:16 aprox)": (512, 768),
        "704 x 480 (Rápido / Calidad estándar)": (704, 480),
        "848 x 480 (Cinemático 16:9)": (848, 480)
    }
    width, height = res_dict.get(resolution, (768, 512))
    
    # Mapeo de cuadros (LTX-Video requiere 8k + 1 cuadros)
    frames_dict = {
        "8 Segundos (~193 frames @ 24fps)": 193,
        "5 Segundos (~121 frames @ 24fps)": 121,
        "10 Segundos (~241 frames @ 24fps)": 241
    }
    num_frames = frames_dict.get(duration_preset, 193)
    fps = 24
    
    # Semilla
    if seed == -1 or seed is None:
        seed = torch.randint(0, 2**32 - 1, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    
    start_time = time.time()
    print(f"🎬 Iniciando render: {width}x{height}, {num_frames} frames ({num_frames/fps:.1f}s), steps={num_inference_steps}, seed={seed}")
    
    video_frames = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt if negative_prompt else None,
        width=width,
        height=height,
        num_frames=num_frames,
        num_inference_steps=int(num_inference_steps),
        guidance_scale=float(guidance_scale),
        generator=generator,
        output_type="np"
    ).frames[0]
    
    elapsed = time.time() - start_time
    
    # Guardar archivo MP4
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"LTX_{timestamp}_seed{seed}.mp4"
    local_path = os.path.join("/content", filename)
    export_to_video(video_frames, local_path, fps=fps)
    
    # Copiar a Drive si existe la carpeta
    drive_path = os.path.join(OUTPUT_DIR, filename)
    if os.path.exists(OUTPUT_DIR):
        import shutil
        shutil.copyfile(local_path, drive_path)
        saved_msg = f"💾 Video guardado en Drive: {drive_path}"
    else:
        saved_msg = f"📁 Guardado localmente en: {local_path}"
        
    status_info = f"✅ Generación completada en {elapsed:.1f} segundos | Frames: {num_frames} ({num_frames/fps:.1f}s) | Seed: {seed}\n{saved_msg}"
    return local_path, status_info

# Construir interfaz con Gradio
with gr.Blocks(title="LTX-Video Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚡ LTX-Video 8-Second High-Speed Studio")
    gr.Markdown("Genera secuencias fluidas de 8 segundos optimizadas para consumo mínimo de cuota en Google Colab Pro.")
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="Prompt (Inglés recomendado para máxima precisión)",
                placeholder="A cinematic slow motion shot of an astronaut walking on glowing neon sand under alien stars, 4k, hyper-detailed...",
                lines=4
            )
            neg_prompt = gr.Textbox(
                label="Negative Prompt",
                placeholder="blurry, low quality, distorted, flickering, artifacts, bad anatomy",
                lines=2
            )
            
            with gr.Row():
                duration = gr.Dropdown(
                    choices=[
                        "8 Segundos (~193 frames @ 24fps)",
                        "5 Segundos (~121 frames @ 24fps)",
                        "10 Segundos (~241 frames @ 24fps)"
                    ],
                    value="8 Segundos (~193 frames @ 24fps)",
                    label="Duración del Video"
                )
                resolution = gr.Dropdown(
                    choices=[
                        "768 x 512 (Horizontal 16:9 aprox)",
                        "512 x 768 (Vertical 9:16 aprox)",
                        "704 x 480 (Rápido / Calidad estándar)",
                        "848 x 480 (Cinemático 16:9)"
                    ],
                    value="768 x 512 (Horizontal 16:9 aprox)",
                    label="Resolución"
                )
            
            with gr.Row():
                steps = gr.Slider(minimum=20, maximum=50, value=30, step=1, label="Inference Steps (30 es óptimo)")
                guidance = gr.Slider(minimum=1.5, maximum=7.5, value=3.5, step=0.5, label="Guidance Scale (CFG)")
            
            seed = gr.Number(value=-1, label="Seed (-1 para aleatorio)", precision=0)
            btn_generate = gr.Button("🚀 Generar Video (8s)", variant="primary")
            
        with gr.Column(scale=1):
            video_output = gr.Video(label="Video Generado", autoplay=True)
            status_box = gr.Textbox(label="Estado de Inferencia", interactive=False)
            
    btn_generate.click(
        fn=generate_video,
        inputs=[prompt, neg_prompt, resolution, duration, steps, guidance, seed],
        outputs=[video_output, status_box]
    )

# Iniciar túnel de Cloudflare en segundo plano
def start_cloudflare_tunnel(port):
    cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tunnel_url = None
    print("⏳ Negociando túnel con Cloudflare Edge Network...")
    for line in iter(p.stdout.readline, ''):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
        if "ERR" in line or "error" in line.lower():
            print(line.strip())
    return tunnel_url, p

cloudflare_url, tunnel_proc = start_cloudflare_tunnel(LOCAL_PORT)
if cloudflare_url:
    print(f"\n🔗 ================================================================")
    print(f"🌐 TU URL PÚBLICA DE CLOUDFLARE:")
    print(f"👉 {cloudflare_url}")
    print(f"🔗 ================================================================\n")
else:
    print("⚠️ No se pudo extraer automáticamente el túnel de Cloudflare. Revisa la consola.")

# Lanzar Gradio en localhost
demo.launch(server_name="0.0.0.0", server_port=LOCAL_PORT, share=False)